# IndustReal: data and annotations

## Section 1 - Purpose

This notebook prepares the public dataset IndustReal (Schoonbeek et al. 2024) for the pipeline of this thesis. IndustReal is the first public test of the Phase 1 pipeline: frozen C-RADIOv4 with prototypes from a small support set. If perception holds, the tracking of Phase 2 is tried on it as it stands.

IndustReal contains 84 egocentric recordings of 27 participants, made with a HoloLens 2, who assemble and maintain a toy car built from 3D-printed parts. The data and the code are released under the Apache 2.0 license. Only the RGB images and the annotations are used here.

The recordings are split by participant into train, validation and test, as in the paper. The support set will come from train, the validation frames from validation, and the test frames from test, so that reference frames never come from recordings used for evaluation.

------------------------------------------------------------------------------------------------------------------------------------------

References Section 1:

Schoonbeek et al. 2024, IndustReal (WACV), Section 4.3: 27 participants; Section 4.4: the split is made on participants, 12 for training, 5 for validation and 10 for testing, to ensure variation in viewpoint, execution and head movements between the three sets.

IndustReal code repository (github.com/TimSchoonbeek/IndustReal), readme: 84 recordings; readme and license.txt: data, code and weights under the Apache 2.0 license.

------------------------------------------------------------------------------------------------------------------------------------------

## Section 2 - Configuration

In [ ]:
# 2.1 Configuration
import json
from pathlib import Path

from remotezip import RemoteZip

# Project folder: the first folder above this notebook that contains src/
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())

# IndustReal on 4TU.ResearchData (doi 10.4121/b008dd74-020d-4ea4-a8ba-7bb60769d224, version 2, Apache-2.0)
DATASET_URL = "https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/"
ZIP_IDS = {                                    # one zip per part of each split, as listed on the dataset page
    "train_p1": "f58a5f18-9a71-4443-8a48-e59530df9ee8",
    "train_p2": "5aae5141-cece-41db-9681-47acd5300705",
    "train_p3": "4e2ed830-a1fe-40fa-9c5a-2aeae3964616",
    "train_p4": "3ff5409e-165f-4354-9fa4-a4fde639a7a6",
    "val_p1": "bb336949-248c-4ae6-82ef-107dbe61d10f",
    "val_p2": "e899de9f-313a-482b-a6b5-e2de4f38412e",
    "test_p1": "ec7a4f94-7c1d-497d-b75a-4e694b7a9f7e",
    "test_p2": "d07789b6-83b1-429d-be0c-95edba210757",
    "test_p3": "31f88cb6-a2e7-44df-ac51-5aaa0a148fc3",
}
ANNOTATION_PREFIXES = ("AR_labels", "OD_labels", "PSR_labels")   # only the annotations; sensor data (gaze, hands, pose) when needed

# List of the 33 procedure steps and the 11 components, from the IndustReal code repository
PROCEDURE_INFO_URL = "https://raw.githubusercontent.com/TimSchoonbeek/IndustReal/ad86a35d4b5125739e24af677e5d7b55c74af945/PSR/procedure_info.json"   # pinned to the last commit (19 Aug 2024)

DATA_DIR = PROJECT_ROOT / "data" / "industreal"       # not tracked by git (data/* in .gitignore)
RECORDINGS_DIR = DATA_DIR / "recordings"              # one folder per split, one subfolder per recording
PROCEDURE_INFO_PATH = DATA_DIR / "procedure_info.json"
MANIFEST_PATH = DATA_DIR / "annotation_manifest.json"   # list of the annotation files downloaded, per zip

print("Project folder:", PROJECT_ROOT)
print("Data folder exists:", DATA_DIR.is_dir())

Project folder: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project
Data folder exists: True


------------------------------------------------------------------------------------------------------------------------------------------

## Section 3 - Annotations

The dataset is published as nine zip files of recordings, three to four per split, of 3.8 to 10.2 GB each. Each recording folder holds every sensor of the headset (RGB, stereo, depth, ambient light), the tracking of gaze, hands and head, and the annotations.

Only the annotation files are needed now. A zip file keeps its index at the end, so the index can be read on the server and single files can be extracted without downloading the whole archive. The sensor data and the RGB frames are downloaded later, and only for the recordings that are used.

The list of the 33 procedure steps and of the 11 components comes from the code repository of the dataset (PSR/procedure_info.json), at its last commit.

In [2]:
# 3.1 Download of the annotations (only the files that are missing)
import urllib.request

def download_annotations():
    """Extract the annotation files of every recording from the remote zips and write the manifest."""
    manifest = {}
    for part, file_id in ZIP_IDS.items():
        split = part.split("_")[0]                                     # train, val or test
        with RemoteZip(DATASET_URL + file_id) as z:                   # reads the zip index on the server
            names = [n for n in z.namelist() if n.split("/")[-1].startswith(ANNOTATION_PREFIXES)]
            for n in names:
                if not (RECORDINGS_DIR / split / n).exists():
                    z.extract(n, RECORDINGS_DIR / split)
        manifest[part] = [f"{split}/{n}" for n in names]
        print(part, len(names), "annotation files")
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=1))

manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else None
if manifest is None or not all((RECORDINGS_DIR / f).exists() for files in manifest.values() for f in files):
    download_annotations()
    manifest = json.loads(MANIFEST_PATH.read_text())
else:
    print("All annotation files already present, no download")
print("Annotation files:", sum(len(files) for files in manifest.values()))

if not PROCEDURE_INFO_PATH.exists():
    urllib.request.urlretrieve(PROCEDURE_INFO_URL, PROCEDURE_INFO_PATH)
print("Procedure info present:", PROCEDURE_INFO_PATH.exists())

All annotation files already present, no download
Annotation files: 420
Procedure info present: True


Observation 3.1: The nine zip files hold 420 annotation files, five for each of the 84 recordings: AR_labels.csv, OD_labels.json, PSR_labels.csv, PSR_labels_with_errors.csv and PSR_labels_raw.csv. The recordings are 36 in train, 16 in validation and 32 in test, with 12, 5 and 10 participants, as in the paper.

Inside the zips the recording folders sit at the top level, without the recordings/train, val and test folders shown in the readme, so the split is taken from the name of the zip. Some image folders also contain Thumbs.db, a Windows system file, which is not part of the data.

The names of the recordings have the form participant_procedure_instruction, for example 05_assy_0_1 or 14_main_2_3: assy is the assembly and main the maintenance. Each participant has one assy_0 and one main_0 recording and one or two more.

------------------------------------------------------------------------------------------------------------------------------------------

References Section 3:

IndustReal code repository (github.com/TimSchoonbeek/IndustReal), readme, Data information and Annotation information: the content of each recording folder and the meaning of the five annotation files; PSR/procedure_info.json, at commit ad86a35 (19 August 2024): the 33 procedure steps and the 11 components.

Schoonbeek et al. 2024, IndustReal (WACV), Section 4.3: each participant performed a correct assembly instruction, a correct maintenance instruction and one or two instructions with errors.

------------------------------------------------------------------------------------------------------------------------------------------